# Aggregate fio read bytes and throughput across ranks

Sums the per-rank `read.io_bytes` and bandwidth from the fio JSON output files
(one file per MPI rank). Rank files may have a non-JSON header line (an MPI/PBS
error message), so parsing starts at the first `{`.

Two aggregate throughput numbers are reported:
- **Sum of per-rank bw**: each rank's own `bw_bytes` summed (optimistic; ignores rank skew).
- **Total bytes / max runtime**: wall-clock view, bounded by the slowest rank.


In [1]:
import json
import glob
from pathlib import Path

# Folder containing one fio JSON file per rank
RESULT_DIR = Path("read_1tib_n1_ppn32_bs2m_iod16")

files = sorted(RESULT_DIR.glob("*.json"))
print(f"{len(files)} rank files")

32 rank files


In [2]:
def load_fio_json(path):
    """Parse a fio JSON file, skipping any non-JSON header lines."""
    txt = Path(path).read_text()
    return json.loads(txt[txt.index("{"):])

per_rank = []
for fp in files:
    d = load_fio_json(fp)
    per_rank.append({
        "file": fp.name,
        "io_bytes": sum(j["read"]["io_bytes"] for j in d["jobs"]),
        "bw_bytes": sum(j["read"]["bw_bytes"] for j in d["jobs"]),
        "runtime_ms": max(j["read"]["runtime"] for j in d["jobs"]),
    })

In [3]:
GiB = 1024**3

total_bytes = sum(r["io_bytes"] for r in per_rank)
total_bw_bytes = sum(r["bw_bytes"] for r in per_rank)
max_rt_s = max(r["runtime_ms"] for r in per_rank) / 1000
min_rt_s = min(r["runtime_ms"] for r in per_rank) / 1000

print(f"Aggregated read bytes    : {total_bytes} B = {total_bytes/GiB:.2f} GiB = {total_bytes/1024**4:.4f} TiB")
print(f"Runtime (min-max)        : {min_rt_s:.1f} - {max_rt_s:.1f} s")
print(f"Sum of per-rank bw       : {total_bw_bytes/GiB:.2f} GiB/s = {total_bw_bytes/1e9:.2f} GB/s")
print(f"Total bytes / max runtime: {total_bytes/max_rt_s/GiB:.2f} GiB/s = {total_bytes/max_rt_s/1e9:.2f} GB/s")

Aggregated read bytes    : 574466555904 B = 535.01 GiB = 0.5225 TiB
Runtime (min-max)        : 8.2 - 9.8 s
Sum of per-rank bw       : 58.54 GiB/s = 62.85 GB/s
Total bytes / max runtime: 54.68 GiB/s = 58.71 GB/s


## Comparison: single-process run (`fio-read1TiB-seqread-fs16g.json`)

A single fio invocation with `numjobs=64`, `size=16gb` per job, `group_reporting=1`
(one JSON file for the whole run), vs. the 32-rank MPI run above.

In [4]:
SINGLE_RUN = Path("fio-read1TiB-seqread-fs16g.json")

d = load_fio_json(SINGLE_RUN)
single = {
    "io_bytes": sum(j["read"]["io_bytes"] for j in d["jobs"]),
    "bw_bytes": sum(j["read"]["bw_bytes"] for j in d["jobs"]),
    "runtime_s": max(j["read"]["runtime"] for j in d["jobs"]) / 1000,
}
print(f"read bytes : {single['io_bytes']} B = {single['io_bytes']/GiB:.2f} GiB = {single['io_bytes']/1024**4:.4f} TiB")
print(f"runtime    : {single['runtime_s']:.1f} s")
print(f"bandwidth  : {single['bw_bytes']/GiB:.2f} GiB/s = {single['bw_bytes']/1e9:.2f} GB/s")

read bytes : 1099511627776 B = 1024.00 GiB = 1.0000 TiB
runtime    : 38.1 s
bandwidth  : 26.85 GiB/s = 28.83 GB/s


In [5]:
rows = [
    ("32-rank MPI run", total_bytes, max_rt_s, total_bw_bytes, total_bytes / max_rt_s),
    ("single-process run", single["io_bytes"], single["runtime_s"], single["bw_bytes"], single["io_bytes"] / single["runtime_s"]),
]

hdr = f"{'run':<20} {'read GiB':>10} {'runtime s':>10} {'sum bw GiB/s':>13} {'wall bw GiB/s':>14}"
print(hdr)
print("-" * len(hdr))
for name, b, rt, bw, wall in rows:
    print(f"{name:<20} {b/GiB:>10.2f} {rt:>10.1f} {bw/GiB:>13.2f} {wall/GiB:>14.2f}")

ratio = (total_bytes / max_rt_s) / (single["io_bytes"] / single["runtime_s"])
print(f"\nMPI run wall-clock bandwidth is {ratio:.2f}x the single-process run")

run                    read GiB  runtime s  sum bw GiB/s  wall bw GiB/s
-----------------------------------------------------------------------
32-rank MPI run          535.01        9.8         58.54          54.68
single-process run      1024.00       38.1         26.85          26.85

MPI run wall-clock bandwidth is 2.04x the single-process run
